## Setup & Data Preparation

In [ ]:
import pandas as pd

print("Loading data...")
calendar = pd.read_csv('data/calendar_events.csv')
train = pd.read_csv('data/train.csv')
submission_sample = pd.read_csv('data/forecast_submission.csv')

print("\n--- Train Data Shape ---")
print(train.shape)

print("\n--- First 5 columns of Train Data ---")
print(train.columns[:5])

Loading data...

--- Train Data Shape ---
(18766, 4)

--- First 5 columns of Train Data ---
Index(['store_id', 'store_name', 'date', 'revenue'], dtype='object')


In [6]:
import pandas as pd

# 1. Merge the train data with the calendar events
df = pd.merge(train, calendar, on='date', how='left')

# 2. Convert 'date' to a proper datetime object so we can extract time features
df['date'] = pd.to_datetime(df['date'])

# 3. Extract basic time features
print("Extracting time features...")
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 4. Create Lag Features (The most important part!)
# We MUST sort by store_id and date first, otherwise our lags will mix up different stores
print("Creating lag features...")
df = df.sort_values(['store_id', 'date']).reset_index(drop=True)

# Create a 7-day lag (revenue exactly 1 week ago)
df['lag_7'] = df.groupby('store_id')['revenue'].shift(7)

# Create a 28-day lag (revenue exactly 4 weeks ago)
df['lag_28'] = df.groupby('store_id')['revenue'].shift(28)

# 5. Handle Calendar Events (Assuming the calendar has a column like 'event_name' or 'event_type')
# Let's check if there's an event today (creates a 1 if there's an event, 0 if not)
# (Note: Check your calendar dataframe for the exact column name for events. It might be 'event_name_1')
if 'event_name' in df.columns:
    df['has_event'] = df['event_name'].notnull().astype(int)
    
# 6. Drop the empty rows created by our shifts
# Since we shifted by 28 days, the first 28 days of data for EVERY store will now have NaNs. We drop them.
df = df.dropna(subset=['lag_28']).reset_index(drop=True)

print(f"Feature engineering complete! Final dataset shape: {df.shape}")
df.head()

Extracting time features...
Creating lag features...
Feature engineering complete! Final dataset shape: (18458, 10)


,store_id,store_name,date,revenue,event,day_of_week,month,is_weekend,lag_7,lag_28
0,0,All Stores,2011-02-26,204598.73,NaN,5,2,1,219127.08,204126.52
1,0,All Stores,2011-02-27,198628.12,NaN,6,2,1,203423.60,197426.42
2,0,All Stores,2011-02-28,146986.65,NaN,0,2,0,163868.92,144267.27
3,0,All Stores,2011-03-01,156350.45,NaN,1,3,0,138451.73,151903.00
4,0,All Stores,2011-03-02,149991.35,NaN,2,3,0,137675.67,117399.88


In [8]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Define Features and Target
# (We exclude 'date', 'revenue', and any non-predictive ID strings)
features = ['store_id', 'day_of_week', 'month', 'is_weekend', 'lag_7', 'lag_28']
if 'has_event' in df.columns:
    features.append('has_event')
    
target = 'revenue'

# Convert categorical columns for LightGBM to understand them natively
df['store_id'] = df['store_id'].astype('category')

# 2. Chronological Train/Validation Split
# We use the absolute last 30 days of the dataset as our validation set
split_date = df['date'].max() - pd.Timedelta(days=30)

train_data = df[df['date'] <= split_date].copy()
valid_data = df[df['date'] > split_date].copy()

X_train, y_train = train_data[features], train_data[target]
X_valid, y_valid = valid_data[features], valid_data[target]

print(f"Training on {len(X_train)} rows...")
print(f"Validating on {len(X_valid)} rows...\n")

# 3. Initialize and Train the Model
model = lgb.LGBMRegressor(
    objective='rmse',          # Our competition evaluation metric!
    n_estimators=1000,         # Maximum number of trees
    learning_rate=0.05,
    random_state=42
)

# Fit the model (Using early stopping so it stops training when validation score stops improving)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

# 4. Check our Local Baseline Score
valid_preds = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))

print(f"\n Local Validation RMSE: {rmse:.2f}")

Training on 18128 rows...
Validating on 330 rows...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000171 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 544
[LightGBM] [Info] Number of data points in the train set: 18128, number of used features: 6
[LightGBM] [Info] Start training from score 43103.212562
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 6456.91	valid_1's rmse: 7556.58
Early stopping, best iteration is:
[119]	training's rmse: 6342.66	valid_1's rmse: 7394.14

 Local Validation RMSE: 7394.14


In [12]:
import pandas as pd
import numpy as np

print("Preparing final submission data...")

# 1. Load the submission sample and calendar
submission = pd.read_csv('data/forecast_submission.csv')
calendar = pd.read_csv('data/calendar_events.csv')

# 🛠️ THE FIX: Convert calendar dates to proper datetime objects so they match!
calendar['date'] = pd.to_datetime(calendar['date'])

# Extract the store_id and date from this ID string
submission['store_id'] = submission['id'].apply(lambda x: int(x.split('_')[0]))
submission['date_str'] = submission['id'].apply(lambda x: x.split('_')[1])

# Convert submission dates to proper datetime objects
submission['date'] = pd.to_datetime(submission['date_str'], format='%Y%m%d')

# 2. Merge with Calendar Events (This will work perfectly now!)
submission = pd.merge(submission, calendar, on='date', how='left')

# Extract future time features
submission['day_of_week'] = submission['date'].dt.dayofweek
submission['month'] = submission['date'].dt.month
submission['is_weekend'] = submission['day_of_week'].isin([5, 6]).astype(int)

if 'has_event' in df.columns: 
    submission['has_event'] = submission['event_name'].notnull().astype(int) if 'event_name' in submission.columns else 0

# 3. The "Cheat" for the Baseline Lags
# Get the last month of real data to use as our baseline anchor
last_known_sales = df.groupby('store_id', observed=True).tail(28) 
avg_recent_sales = last_known_sales.groupby('store_id', observed=True)['revenue'].mean().reset_index()
avg_recent_sales.rename(columns={'revenue': 'recent_avg'}, inplace=True)

submission = pd.merge(submission, avg_recent_sales, on='store_id', how='left')

# Fill our lag features with the recent average
submission['lag_7'] = submission['recent_avg']
submission['lag_28'] = submission['recent_avg']

# Convert store_id to categorical (LightGBM requires this)
submission['store_id'] = submission['store_id'].astype('category')

# 4. Make Predictions!
print("Generating predictions...")
features = ['store_id', 'day_of_week', 'month', 'is_weekend', 'lag_7', 'lag_28']
if 'has_event' in df.columns:
    features.append('has_event')

submission['prediction'] = model.predict(submission[features])

# 5. Format and Save the Final File
final_output = submission[['id', 'prediction']]
final_output.to_csv('my_first_submission.csv', index=False)

print("'my_first_submission.csv' has been saved successfully.")
print("\nSubmission preview:")
print(final_output.head())

Preparing final submission data...
Generating predictions...
'my_first_submission.csv' has been saved successfully.

Submission preview:
           id     prediction
0  0_20151001  277824.233558
1  0_20151002  285762.937271
2  0_20151003  290854.478294
3  0_20151004  290012.402218
4  0_20151005  282749.900806
